# 2 — Build the reduced data

The raw corpora are ~8 GB. Every figure and table in the paper is computed from
a few hundred kilobytes of numeric summaries, and this notebook builds them.

The reduction is not a convenience: it is what makes the analysis checkable. The
artifacts in `data_reduced/` are **numbers only** — frequency vectors, spectra,
network degree and strength, vocabulary-growth curves — with no strings, so they
carry no redistributable text from any of the three corpora.

| artifact | contents |
| --- | --- |
| `spgc_<lang>_1gram.npz` | aggregated 1-gram counts, vocabulary, token total |
| `spgc_<lang>_ngram.npz` | frequency-of-frequency for orders k = 1..5 |
| `spgc_<lang>_wcn.npz` | word co-occurrence network: strength, degree, node and edge counts |
| `spgc_<lang>_heaps.npz` | vocabulary growth D(t), mean and 5–95% band over random book orders |
| `spgc_<lang>_phrases.json` | occurrence counts of the annotated concepts (the only artifact with strings) |
| `corefl_<group>.npz` | the same, per COREFL population |
| `parseme_mwe.npz` | the MWE and n-gram spectra of Table S9 |

Two construction details matter and are shared by every artifact: n-grams are
counted **within books** and never across book boundaries, and the co-occurrence
network links tokens adjacent within a book, **excluding self-loops** (0.45% of
English bigram positions).

In [ ]:
import os
import subprocess
import sys

REPO = os.path.abspath("..") if os.path.isdir(os.path.join("..", "src")) else os.path.abspath(".")
sys.path.insert(0, os.path.join(REPO, "src"))

import plotting as P


def run(*command, must_succeed=True):
    """Run one pipeline step and, unlike a `!` cell, STOP if it fails.

    An IPython `!` cell throws away the exit status: a step that dies leaves no
    output, no error and no trace, and `nbconvert --execute` still reports the
    notebook as successful. Two defects in this pipeline's history hid exactly
    there, so every step below goes through this instead.

    `must_succeed=False` is used only for the two `--check` diagnostics of
    notebook 1, which print a loud banner rather than stopping the run.
    """
    print(">>", " ".join(str(c) for c in command), flush=True)
    code = subprocess.run([str(c) for c in command], check=False).returncode
    if code and must_succeed:
        raise RuntimeError(f"step failed with exit code {code} - read the output "
                           f"above; nothing after this point is valid")
    if code:
        rule = "*" * 72
        print(rule)
        print(f"*** THIS CHECK FAILED (exit code {code}). Read the output above")
        print("*** before going on: whatever depends on this corpus is missing")
        print("*** or wrong, and so is anything computed from it.")
        print(rule, flush=True)
    return code


def py(script, *args, must_succeed=True):
    """`run` for one of this repository's own scripts."""
    return run(sys.executable, os.path.join(REPO, "src", script), *args,
               must_succeed=must_succeed)


%matplotlib inline
USETEX = P.setup_style()
print("repo:", REPO, "| LaTeX text rendering:", USETEX)

## 2.1 SPGC

Two passes over the raw bulk. The zips are streamed in place, never extracted.
The `--counts` pass is minutes; `--tokens` is roughly half an hour and is the
one that builds the n-gram spectra and the networks. N-grams are counted with
64-bit hashing, which keeps peak memory near 2 GB even for the 170 M-token
languages and is exact up to negligible collisions.

In [ ]:
py("build_reduced.py", "--counts")

In [ ]:
py("build_reduced.py", "--tokens")

Vocabulary growth curves D(t), averaged over random orderings of the books and
carried up to 10^8 tokens — the length every model comparison uses. A language
whose corpus is shorter than that stops at its own total, so English and French
reach 10^8 while German, Italian and Spanish end at 72.1, 42.9 and 36.0 M.

The four values below are not defaults and all four matter: `--heaps-max-tokens`
sets the upper end of the curve, `--match-tokens` the length of the size-matched
frequency vector, `--heaps-points` its resolution and `--shuffles` the width of
the 5-95% band. Note that `--max-tokens` is a *different* option belonging to
`--tokens`; passing it here changes nothing and silently leaves the curves at
the 10^6 default.

In [ ]:
py("build_reduced.py", "--heaps", "--heaps-max-tokens", "100000000", "--match-tokens", "100000000", "--heaps-points", "120", "--shuffles", "30")

## 2.2 The annotated concepts

The reduced n-gram data is hashed and identity-free, which is what keeps it
small. The concepts annotated on Figure 1 are specific strings, so their counts
have to be measured once from the raw tokens. This is the only step besides 2.1
that reads `data_raw/`.

The concept list lives in `manifests/annotation_phrases.json` and is discussed
in notebook 5: it was chosen by hand and fixed **before** any count was looked
at.

In [ ]:
py("count_phrases.py")

## 2.3 COREFL

Same two-step pattern, on the learner corpus. Both outputs are numeric only, so
no COREFL text is redistributed and `data_raw/corefl/` can be deleted afterwards.

In [ ]:
py("build_reduced_corefl.py")

## 2.4 PARSEME

Reduces the annotated MWEs and the surface n-grams of the same gold text to
frequency spectra — a few kB, from which Table S9 is recomputed without either
corpus. Skip this cell if you did not download PARSEME; only Table S9 depends
on it.

In [ ]:
py("mwe_ranks.py", "--build")

## 2.5 Check: the composition of the five corpora

This is **Table S1**. Every number is read back from `data_reduced/`, i.e. from
the data the figures are actually computed on, rather than from the manifest —
so if a manifest book is missing from the frozen release, the table says so.

In [ ]:
import corpus_table

table = corpus_table.build()
corpus_table.write_table(table)
table

## What you have now

`data_reduced/` is complete. One more notebook still reads `data_raw/` —
notebook 3, which resamples the raw token streams for Table S3 and the length
controls. After that nothing does: notebooks 4 to 7 and 9 read only
`data_reduced/` and the tables written along the way, and notebook 8 runs the
simulator.

**Next:** `03_corpus_controls.ipynb`.